# Skew40 Distributional-Similarity Test

**Question:** does relative skill (MAE_Chronos / MAE_Panda) correlate with how close an
evaluation system is to Panda's skew40 pretraining distribution?

**Status:** last standing candidate mechanism after four negative results (channel
attention, Koopman-lift geometry, temporal attention, resolution-dependency).

**Two arms, pre-registered before either is computed:**
- **Arm 1 (representation-space distance):** distance from each eval system's Panda
  feature centroid to the skew40 reference set, in Panda's own lift feature space.
  Exposed to a circularity concern (Panda's own representation predicting Panda's own
  accuracy is not fully independent evidence) — this is *why* Arm 2 exists.
- **Arm 2 (model-agnostic distance):** distance computed from raw dynamical descriptors
  that never touch Panda's weights, so it cannot be circular by construction.

**Decision rule (fixed here, before any number is computed):**
- Support for H-dist requires **negative** Spearman correlation (closer to skew40 →
  higher relative skill) with **|ρ| ≥ 0.5** in at least one arm.
- **Strong** support requires both arms to agree in sign.
- Disagreement between arms is read as evidence of circularity in Arm 1, not as
  support for H-dist.

**Hard rule: the system list below is frozen. Do not add or remove systems after
seeing a correlation number** — this is exactly the failure mode that invalidated
Experiment 4 (dysts p-hacking) earlier in this project.

**Provenance of this notebook's filled-in cells:** model loading is adapted from
`fixed_experiments.ipynb` (Cell 0, published-checkpoint loading — this experiment
deliberately uses the published `GilpinLab/panda` checkpoint, not `baseline_100k`,
since it's testing a property of the model that actually produced the logged
advantage numbers). Forward-hook logic is adapted from `a3_koopman_geometry.ipynb`
(Section 3, auto-discovery of the lift module + inner projection Linear). Eval-system
simulators (Lorenz `gate_3ch`, Rossler, SprottB, Van der Pol, Duffing, Harmonic,
Burgers PCA sweep, Weather) are adapted from `eval-nb.ipynb` (Sections 5, 6, 8) and
`fixed_experiments.ipynb`'s original Burgers sweep. skew40 loading uses
`eval-nb.ipynb`'s `load_dataset('GilpinLab/skew40', split='train')` call, extended
with the `_np_shape` trajectory-array reconstruction described in project conventions
— **this extension is written from documented convention, not verified against a live
schema, and is flagged for confirmation in Cell 6.**

**A resolved protocol discrepancy, stated up front:** `eval-nb.ipynb`'s Burgers
ν=1.0 loader uses `T=1500` (Experiment 28/A3's OOD-specific protocol);
`fixed_experiments.ipynb`'s original sweep (the actual source of every `rel_skill`
number in the frozen table below, ν=1.0 included) uses `T=1000`. This notebook uses
**T=1000 throughout**, to match the protocol that actually produced the numbers being
correlated against, not the later OOD-specific variant.


## What's filled in vs. what you still need to confirm

**Filled in, from your notebooks:**
- Cell 3: imports
- Cell 4: model loading (published Panda checkpoint)
- Cell 5: forward hooks (Φ_pre / Φ_post auto-discovery, from A3)
- Cell 6: skew40 loading — **extraction logic needs a schema check, see the TODO inside**
- Cell 9: all 15 eval-system context-window generators

**Still needs a decision from you:**
- `FEATURE_SPACE` in Cell 5 (defaults to `"pre"` — confirm or override)
- `SKEW40_SAMPLE_N` in Cell 6 (defaults to 300 — confirm or override)
- Confirm Cell 6's `_np_shape` extraction actually matches the live skew40 schema
  before trusting Cell 7's output (the notebook will fail loudly if the column names
  are wrong, but silent mis-extraction, e.g. wrong axis order, is possible and
  worth a manual spot-check on one row).


In [1]:
# ============================================================
# CELL 3 — IMPORTS
# ============================================================
import os
import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr
from scipy.spatial.distance import mahalanobis
from scipy.integrate import solve_ivp
from scipy.fft import fft, ifft, fftfreq
from scipy.linalg import svd

import sys
sys.path.insert(0, './panda')  # local run: panda repo cloned in the working directory
from panda.patchtst.pipeline import PatchTSTPipeline

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device={device}')

CONTEXT_LEN = 512

# Confirm before running:
#   - transformers==4.40.2 active in this kernel
#   - peft NOT installed
#   - kernel restarted after any Cell-1 numpy binary-conflict fix, if a fresh Kaggle session


C:\Users\user\panda_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device=cpu


In [2]:
# ============================================================
# CELL 4 — LOAD PANDA MODEL (published checkpoint)
# Adapted from fixed_experiments.ipynb, Cell 0.
# ============================================================
panda_pipe = PatchTSTPipeline.from_pretrained(
    mode="predict",
    pretrain_path="GilpinLab/panda",
    device_map=device,
)
model = panda_pipe.model
model.eval()
print('Loaded published GilpinLab/panda checkpoint.')


C:\Users\user\panda_env\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loaded published GilpinLab/panda checkpoint.


In [3]:
# ============================================================
# CELL 5 — FORWARD HOOKS FOR FEATURE EXTRACTION
# Adapted verbatim (auto-discovery logic) from a3_koopman_geometry.ipynb, Section 3.
# ============================================================
FEATURE_SPACE = "pre"  # TODO: confirm or change to "post". Default: "pre",
                        # since Phi_post is a learned projection fit to the
                        # forecasting objective and is more exposed to the
                        # Arm-1 circularity concern than Phi_pre.

# --- Round 2: the previous keyword search was wrong on two counts --
#   1. 'dict' is a substring of 'predict'/'prediction', so it false-positive
#      matched the root model (PatchTSTForPrediction) and the head
#      (PatchTSTPredictionHead) -- neither is the lift.
#   2. The real target was never in the keyword list: A3 (Section 8) directly
#      confirmed via source inspection that the class is named
#      PatchTSTKernelEmbedding. 'embed'/'kernel' were missing entirely.
# Given two wrong auto-detections in a row, this now prints the FULL module
# tree and requires an exact-class-name match (specific, low false-positive
# risk) before proceeding; broader keyword matching is only a fallback, and
# does NOT auto-select -- it requires you to set DYNAMICS_EMBED_MODULE_NAME
# by hand after reading the printed candidates.

print('Full module tree:\n')
all_named_modules = list(model.named_modules())
for name, mod in all_named_modules:
    print(f'  {name!r:50s} {mod.__class__.__name__}')

EXACT_CLASS_MATCH = 'PatchTSTKernelEmbedding'  # per A3 Section 8, confirmed via source
exact_candidates = [(name, mod) for name, mod in all_named_modules
                     if mod.__class__.__name__ == EXACT_CLASS_MATCH]

DYNAMICS_EMBED_MODULE_NAME = None
target_module = None

if exact_candidates:
    DYNAMICS_EMBED_MODULE_NAME, target_module = exact_candidates[0]
    print(f"\nExact class match found: '{EXACT_CLASS_MATCH}' at '{DYNAMICS_EMBED_MODULE_NAME}'. Using this module.")
    if len(exact_candidates) > 1:
        print(f'[WARNING] {len(exact_candidates)} modules share this class name -- using the first '
              f'({DYNAMICS_EMBED_MODULE_NAME!r}). Others: {[n for n, _ in exact_candidates[1:]]}. '
              'Confirm this is the right one before trusting downstream results.')
else:
    print(f"\nNo module with class name '{EXACT_CLASS_MATCH}' found. Broadened (corrected) keyword "
          f"candidates below -- 'dict' removed (too broad), 'embed'/'kernel' added:")
    candidate_modules = []
    for name, mod in all_named_modules:
        lname, cname = name.lower(), mod.__class__.__name__.lower()
        if any(k in lname or k in cname for k in
               ['koopman', 'kernel', 'dynamics', 'lift', 'rff', 'poly', 'embed']):
            candidate_modules.append((name, mod.__class__.__name__))
            print(f'  candidate: {name!r:50s} {mod.__class__.__name__}')
    print('\nSET DYNAMICS_EMBED_MODULE_NAME MANUALLY BELOW based on the full tree/candidates above -- '
          'this notebook will not guess a third time.')
    # TODO if this branch fired: DYNAMICS_EMBED_MODULE_NAME = '...'
    #      target_module = dict(all_named_modules)[DYNAMICS_EMBED_MODULE_NAME]

assert DYNAMICS_EMBED_MODULE_NAME is not None and target_module is not None, (
    'Manual selection required -- see the printed module tree above, then set '
    "DYNAMICS_EMBED_MODULE_NAME and target_module by hand, then re-run this cell."
)
print(f'\nHooking module: {DYNAMICS_EMBED_MODULE_NAME}')

_captured = {'pre': None, 'post': None}

def _hook_fn(module, inputs, output):
    _captured['post'] = output.detach()
    if hasattr(module, 'last_dict'):
        _captured['pre'] = module.last_dict.detach()

handle_post = target_module.register_forward_hook(_hook_fn)

inner_linear_name, inner_linear = None, None
for name, mod in target_module.named_modules():
    if isinstance(mod, torch.nn.Linear):
        inner_linear_name, inner_linear = name, mod
        break  # first Linear inside the lift is almost certainly the dict->d_model projection

def _pre_hook_fn(module, inputs):
    _captured['pre'] = inputs[0].detach()

handle_pre = None
if inner_linear is not None:
    handle_pre = inner_linear.register_forward_pre_hook(_pre_hook_fn)
    print(f'Found inner projection Linear at "{DYNAMICS_EMBED_MODULE_NAME}.{inner_linear_name}" '
          f'(in_features={inner_linear.in_features}, out_features={inner_linear.out_features}) '
          '-- hooking its input as Phi_pre.')
else:
    print('[WARNING] No inner nn.Linear found inside the lift module. Phi_pre will only be '
          'populated if the module sets a `.last_dict` attribute during forward(). Inspect '
          'the module source before trusting Phi_pre results:')
    print(target_module)

def extract_features(context_window):
    """
    context_window: np.ndarray, shape (num_channels, time) -- i.e. (C, T), matching
    this project's convention everywhere else (skew40 schema, Burgers/Lorenz/etc.
    loaders in Cell 9). Panda's PatchTSTModel.forward() requires the OPPOSITE
    order, (batch, sequence_length, num_channels), confirmed directly from source
    in A3 (Section 8, "Input-orientation error"). Transposing here, not upstream,
    so every other cell in this notebook can keep using (C, T) consistently and
    this is the one place the flip happens.
    """
    x = torch.as_tensor(context_window.T, dtype=torch.float32, device=device).unsqueeze(0)  # (1, T, C)
    with torch.no_grad():
        _ = model(x)
    feat = _captured['pre'] if FEATURE_SPACE == "pre" else _captured['post']
    assert feat is not None, (
        f'FEATURE_SPACE="{FEATURE_SPACE}" was requested but hook captured None. '
        'Run the Cell 4-equivalent sanity check from A3 (Section 4) before proceeding.'
    )
    feat_np = feat.squeeze(0).cpu().numpy().reshape(-1, feat.shape[-1])
    return feat_np.mean(axis=0)  # mean-pool across patches -> 1D vector


Full module tree:

  ''                                                 PatchTSTForPrediction
  'model'                                            PatchTSTModel
  'model.scaler'                                     PatchTSTScaler
  'model.scaler.scaler'                              PatchTSTStdScaler
  'model.patchifier'                                 PatchTSTPatchify
  'model.masking'                                    Identity
  'model.encoder'                                    PatchTSTEncoder
  'model.encoder.embedder'                           PatchTSTKernelEmbedding
  'model.encoder.embedder.projection'                Linear
  'model.encoder.layers'                             ModuleList
  'model.encoder.layers.0'                           PatchTSTEncoderLayerWithRope
  'model.encoder.layers.0.temporal_self_attn'        PatchTSTRopeAttention
  'model.encoder.layers.0.temporal_self_attn.k_proj' Linear
  'model.encoder.layers.0.temporal_self_attn.v_proj' Linear
  'model.encoder.laye

In [4]:
# ============================================================
# CELL 6 — LOAD SKEW40 REFERENCE SAMPLE
# hf_dataset load is verbatim from eval-nb.ipynb, Section 6 (Cell 10).
# Trajectory-array extraction: CONFIRMED against real schema (previously a guess,
# now fixed). Real columns: 'start', 'target._np_shape' (literal dot in the key
# string, NOT nested attribute access), 'target' (flat float64 list),
# '_source_directory', '_source_filename'. Confirmed example: target has len=12288,
# target._np_shape=[3, 4096] (3 channels x 4096 timesteps, channels-first, matching
# this project's (channels, time) convention throughout).
# ============================================================
try:
    from datasets import load_dataset
except ImportError:
    import subprocess, sys as _sys
    print("'datasets' not found -- installing (this is a one-time setup, ~10-20s)...")
    subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'datasets'], check=True)
    from datasets import load_dataset

hf_dataset = load_dataset('GilpinLab/skew40', split='train')

SKEW40_SAMPLE_N = 1000  # bumped from 300: Phi_pre is ~392-dim, and 300 samples
                          # leaves the covariance rank-deficient before any
                          # regularization. 1000 is still a small fraction of
                          # skew40's ~21k trajectories.
SKEW40_SEED = 0

print(f'skew40 full split: {len(hf_dataset)} rows')
print(f'Columns: {hf_dataset.column_names}')

rng = np.random.default_rng(SKEW40_SEED)
sample_idx = rng.choice(len(hf_dataset), size=min(SKEW40_SAMPLE_N, len(hf_dataset)), replace=False)
skew40_subset = hf_dataset.select(sample_idx.tolist())
skew40_df = skew40_subset.to_pandas()

def _reconstruct_trajectory(row, verbose=False):
    """Confirmed schema: row['target'] is a flat float64 list, row['target._np_shape']
    is a 2-element [dim0, dim1] list. Axis order is inferred per-row (smaller dim =
    channels, since chaotic-system channel counts are always small and trajectory
    length is always large) rather than assumed globally, in case it varies across
    skew-product source systems."""
    flat = np.asarray(row['target'], dtype=np.float64)
    shape = tuple(int(s) for s in row['target._np_shape'])
    assert np.prod(shape) == len(flat), (
        f"shape {shape} (product={np.prod(shape)}) doesn't match flat length {len(flat)} "
        f"for source={row.get('_source_directory')}"
    )
    arr = flat.reshape(shape)
    if shape[0] > shape[1]:
        arr = arr.T  # flip to channels-first if the smaller dim came second
        if verbose:
            print(f"  transposed: raw shape {shape} -> channels-first {arr.shape}")
    return arr

# Sanity check on the first few rows before committing to the full sample
print("\nSanity check, first 3 rows:")
for _, row in skew40_df.head(3).iterrows():
    traj = _reconstruct_trajectory(row, verbose=True)
    print(f"  source={row['_source_directory']:30s} raw_shape={tuple(row['target._np_shape'])} "
          f"-> final_shape={traj.shape}")

skew40_trajectories = [_reconstruct_trajectory(row) for _, row in skew40_df.iterrows()]
print(f'\nReconstructed {len(skew40_trajectories)} skew40 trajectories.')

# Trim to at least CONTEXT_LEN steps; drop trajectories shorter than that rather
# than pad, since a padded window would not be a genuine sample of skew40 dynamics.
before = len(skew40_trajectories)
skew40_trajectories = [t for t in skew40_trajectories if t.shape[-1] >= CONTEXT_LEN]
print(f'{len(skew40_trajectories)}/{before} trajectories retain >= {CONTEXT_LEN} steps and are usable.')
skew40_trajectories = [t[:, -CONTEXT_LEN:] for t in skew40_trajectories]  # take last CONTEXT_LEN steps


skew40 full split: 20979 rows
Columns: ['start', 'target._np_shape', 'target', '_source_directory', '_source_filename']

Sanity check, first 3 rows:
  source=SprottN_PanXuZhou              raw_shape=(3, 4096) -> final_shape=(3, 4096)
  source=Finance_HyperPang              raw_shape=(4, 4096) -> final_shape=(4, 4096)
  source=Laser_StickSlipOscillator      raw_shape=(3, 4096) -> final_shape=(3, 4096)

Reconstructed 1000 skew40 trajectories.
1000/1000 trajectories retain >= 512 steps and are usable.


In [5]:
# ============================================================
# CELL 7 — EXTRACT ARM 1 FEATURES FOR SKEW40 REFERENCE SET
# (fully written; runs once Cells 4-6 are confirmed working)
# ============================================================
skew40_features = np.array([extract_features(traj) for traj in skew40_trajectories])
print(f"skew40 reference set: {skew40_features.shape[0]} trajectories, "
      f"{skew40_features.shape[1]}-dim features (space={FEATURE_SPACE})")

skew40_centroid = skew40_features.mean(axis=0)
skew40_cov = np.cov(skew40_features, rowvar=False)
skew40_cov_reg = skew40_cov + 1e-4 * np.eye(skew40_cov.shape[0])
skew40_cov_inv = np.linalg.pinv(skew40_cov_reg)


skew40 reference set: 1000 trajectories, 512-dim features (space=pre)


# ============================================================
# CELL 7 — EXTRACT ARM 1 FEATURES FOR SKEW40 REFERENCE SET
# (fully written; runs once Cells 4-6 are confirmed working)
# Covariance: Ledoit-Wolf shrinkage instead of ad hoc epsilon regularization --
# with SKEW40_SAMPLE_N=1000 and Phi_pre ~392-dim, sample covariance is still
# poorly conditioned; shrinkage toward a scaled identity is the standard fix,
# not a compute-budget one.
# ============================================================
from sklearn.covariance import LedoitWolf

skew40_features = np.array([extract_features(traj) for traj in skew40_trajectories])
print(f"skew40 reference set: {skew40_features.shape[0]} trajectories, "
      f"{skew40_features.shape[1]}-dim features (space={FEATURE_SPACE})")

skew40_centroid = skew40_features.mean(axis=0)

lw = LedoitWolf().fit(skew40_features)
skew40_cov_inv = np.linalg.inv(lw.covariance_)
print(f"Ledoit-Wolf shrinkage coefficient: {lw.shrinkage_:.4f} "
      f"(0 = no shrinkage / raw sample covariance, 1 = fully shrunk to scaled identity)")
if lw.shrinkage_ > 0.5:
    print("[NOTE] High shrinkage -- the raw sample covariance was poorly conditioned even "
          "at this sample size. Distances are still usable but treat Arm 1 with a bit more "
          "caution than a low-shrinkage result would warrant.")


In [6]:
# ============================================================
# CELL 8 — FROZEN RELATIVE-SKILL TABLE (fully written, do not edit post-hoc)
# ============================================================
FROZEN_SYSTEMS = pd.DataFrame([
    # system_key,      rel_skill, n_windows, source
    ("lorenz",          9.60,  8,  "Exp19 revision"),
    ("harmonic",        6.70,  8,  "Exp19 revision"),
    ("rossler",         4.70,  8,  "Exp19 revision"),
    ("burgers_nu1p0",   3.06,  8,  "Exp10 revision"),
    ("burgers_nu0p5",   2.54,  8,  "Exp10 revision"),
    ("burgers_nu0p05",  2.09,  8,  "Exp10 revision"),
    ("burgers_nu0p1",   1.99,  8,  "Exp10 revision"),
    ("burgers_nu0p005", 1.81,  8,  "Exp10 revision"),
    ("burgers_nu0p02",  1.86,  8,  "Exp10 revision"),
    ("burgers_nu0p01",  1.69,  8,  "Exp10 revision"),
    ("weather_h96",     1.272, 20, "Exp8"),
    ("weather_h192",    1.326, 20, "Exp8"),
    ("weather_h336",    1.279, 20, "Exp8"),
    ("duffing",         1.37,  8,  "Exp19 revision"),
    ("van_der_pol",     1.33,  8,  "Exp19 revision"),
    ("burgers_nu2p0",   1.26,  8,  "Exp10 revision (n.s.)"),
], columns=["system_key", "rel_skill", "n_windows", "source"])

print(FROZEN_SYSTEMS.to_string(index=False))


     system_key  rel_skill  n_windows                source
         lorenz      9.600          8        Exp19 revision
       harmonic      6.700          8        Exp19 revision
        rossler      4.700          8        Exp19 revision
  burgers_nu1p0      3.060          8        Exp10 revision
  burgers_nu0p5      2.540          8        Exp10 revision
 burgers_nu0p05      2.090          8        Exp10 revision
  burgers_nu0p1      1.990          8        Exp10 revision
burgers_nu0p005      1.810          8        Exp10 revision
 burgers_nu0p02      1.860          8        Exp10 revision
 burgers_nu0p01      1.690          8        Exp10 revision
    weather_h96      1.272         20                  Exp8
   weather_h192      1.326         20                  Exp8
   weather_h336      1.279         20                  Exp8
        duffing      1.370          8        Exp19 revision
    van_der_pol      1.330          8        Exp19 revision
  burgers_nu2p0      1.260          8 Ex

In [7]:
# ============================================================
# CELL 9 — EVAL-SYSTEM CONTEXT WINDOWS
# Simulators adapted from eval-nb.ipynb (Lorenz gate_3ch, Rossler, SprottB,
# Van der Pol, Duffing, Harmonic, Burgers/pca_reduction) and fixed_experiments.ipynb
# (Burgers sweep at T=1000, Weather via load_ts). All use SEED for reproducibility.
# ============================================================

DATA_DIR = './ts_data'  # local run: ts_data in the working directory, matching g5_chronos_horizon_mismatch.ipynb

N_WINDOWS_PER_SYSTEM = 20  # bumped from 1: fixes window-sampling noise per system.
                            # Does NOT fix single-trajectory-per-system limitation --
                            # see N_SEEDS_PER_SYSTEM below for that separate axis.
N_SEEDS_PER_SYSTEM = 1      # TODO: bump if you also want multiple independently-seeded
                            # trajectories per system, not just multiple windows within
                            # one trajectory. Cheap to raise (e.g. to 3-5) given overnight
                            # budget, but changes aggregation below (would need a seed loop
                            # around each load_* call) -- left at 1 as a default, not a
                            # silent assumption that 1 seed is sufficient.

def extract_context_windows(trajectory_CT, n_windows=N_WINDOWS_PER_SYSTEM, context_len=CONTEXT_LEN):
    """trajectory_CT: (channels, time). Returns list of (channels, context_len)
    windows, start positions chosen via linspace, matching this project's standard
    window-selection convention (evaluate()/single_condition_mae())."""
    C, T = trajectory_CT.shape
    if T < context_len:
        raise ValueError(f"trajectory too short: T={T} < context_len={context_len}")
    max_start = T - context_len
    starts = np.linspace(0, max_start, n_windows, dtype=int) if max_start > 0 else [0]
    return [trajectory_CT[:, s:s+context_len] for s in starts]

# --- Lorenz (gate_3ch protocol, verbatim from eval-nb.ipynb Section 5) ---
def simulate_lorenz_gate(n=5000, dt=0.01, sigma=10, rho=28, beta=8/3):
    x, y, z = 0.1, 0.0, 0.0
    xs, ys, zs = [x], [y], [z]
    for _ in range(n - 1):
        k1x = sigma * (y - x); k1y = x * (rho - z) - y; k1z = x * y - beta * z
        k2x = sigma * ((y + dt/2*k1y) - (x + dt/2*k1x))
        k2y = (x + dt/2*k1x) * (rho - (z + dt/2*k1z)) - (y + dt/2*k1y)
        k2z = (x + dt/2*k1x) * (y + dt/2*k1y) - beta * (z + dt/2*k1z)
        k3x = sigma * ((y + dt/2*k2y) - (x + dt/2*k2x))
        k3y = (x + dt/2*k2x) * (rho - (z + dt/2*k2z)) - (y + dt/2*k2y)
        k3z = (x + dt/2*k2x) * (y + dt/2*k2y) - beta * (z + dt/2*k2z)
        k4x = sigma * ((y + dt*k3y) - (x + dt*k3x))
        k4y = (x + dt*k3x) * (rho - (z + dt*k3z)) - (y + dt*k3y)
        k4z = (x + dt*k3x) * (y + dt*k3y) - beta * (z + dt*k3z)
        x += dt/6*(k1x+2*k2x+2*k3x+k4x)
        y += dt/6*(k1y+2*k2y+2*k3y+k4y)
        z += dt/6*(k1z+2*k2z+2*k3z+k4z)
        xs.append(x); ys.append(y); zs.append(z)
    return np.array([xs, ys, zs]).T

def load_lorenz():
    traj = simulate_lorenz_gate(n=5000)[500:3500].T  # (3, 3000)
    return traj

# --- Rossler, SprottB (verbatim from eval-nb.ipynb Section 6) ---
def simulate_rossler(n_steps=4000, a=0.2, b=0.2, c=5.7, seed=SEED):
    rng = np.random.default_rng(seed)
    def rhs(t, y):
        return [-y[1]-y[2], y[0]+a*y[1], b+y[2]*(y[0]-c)]
    ic = rng.standard_normal(3)
    sol = solve_ivp(rhs, [0, n_steps*0.05], ic,
                     t_eval=np.linspace(0, n_steps*0.05, n_steps),
                     method='RK45', rtol=1e-9, atol=1e-9)
    return sol.y  # (3, n_steps)

def simulate_sprott_b(n_steps=4000, seed=SEED):
    rng = np.random.default_rng(seed)
    def rhs(t, state):
        x, y, z = state
        return [y*z, x - y, 1 - x*y]
    ic = rng.standard_normal(3)
    sol = solve_ivp(rhs, [0, n_steps*0.05], ic,
                     t_eval=np.linspace(0, n_steps*0.05, n_steps),
                     method='RK45', rtol=1e-9, atol=1e-9)
    return sol.y  # (3, n_steps)

def load_rossler():
    return simulate_rossler(n_steps=4000, seed=SEED)[:, 500:]  # (3, 3500)

def load_sprottb():
    return simulate_sprott_b(n_steps=4000, seed=SEED)[:, 500:]  # (3, 3500) -- kept for reference,
    # not in FROZEN_SYSTEMS (SprottB has no logged rel_skill number in the summary
    # tables this project's log reports advantage as; excluded from the frozen table
    # for that reason, not because the trajectory generator is unavailable).

# --- Van der Pol, Duffing, Harmonic (verbatim from eval-nb.ipynb Section 8) ---
def simulate_harmonic(n_steps=4000, omega=1.0, seed=SEED):
    rng = np.random.default_rng(seed)
    dt = 0.05
    x, v = float(rng.standard_normal()), float(rng.standard_normal())
    traj = []
    for _ in range(n_steps):
        traj.append(x)
        x_new = x + v * dt
        v_new = v - omega**2 * x * dt
        x, v = x_new, v_new
    return np.array(traj, dtype=np.float32)

def simulate_vanderpol(n_steps=4000, mu=2.0, seed=SEED):
    rng = np.random.default_rng(seed)
    def vdp(t, y):
        return [y[1], mu*(1 - y[0]**2)*y[1] - y[0]]
    ic = rng.standard_normal(2).tolist()
    sol = solve_ivp(vdp, [0, n_steps*0.05], ic,
                     t_eval=np.linspace(0, n_steps*0.05, n_steps),
                     method='RK45', rtol=1e-8, atol=1e-8)
    return sol.y[0].astype(np.float32)

def simulate_duffing(n_steps=4000, delta=0.3, alpha=-1.0, beta=1.0, gamma=0.37, omega=1.2, seed=SEED):
    rng = np.random.default_rng(seed)
    dt = 2*np.pi / omega / 50
    x, v = float(rng.standard_normal()), float(rng.standard_normal())
    traj = []
    t = 0.0
    for _ in range(n_steps):
        traj.append(x)
        ax = -delta*v - alpha*x - beta*x**3 + gamma*np.cos(omega*t)
        x_new = x + v*dt
        v_new = v + ax*dt
        x, v, t = x_new, v_new, t+dt
    return np.array(traj, dtype=np.float32)

def load_harmonic():
    return simulate_harmonic(n_steps=4000, omega=1.0, seed=SEED)[500:][None, :]  # (1, 3500)

def load_vanderpol():
    return simulate_vanderpol(n_steps=4000, mu=2.0, seed=SEED)[500:][None, :]  # (1, 3500)

def load_duffing():
    return simulate_duffing(n_steps=4000, seed=SEED)[500:][None, :]  # (1, 3500)

# --- Burgers PCA sweep, T=1000 (matching fixed_experiments.ipynb's original sweep,
#     the actual source of every burgers_nu* row in FROZEN_SYSTEMS) ---
def simulate_burgers_stable(T=1000, N_x=128, nu=0.005, seed=SEED):
    rng = np.random.default_rng(seed)
    dx = 2 * np.pi / N_x
    dt_diff = 0.4 * dx**2 / (2 * nu + 1e-10)
    dt_adv = 0.4 * dx
    dt = min(dt_diff, dt_adv, 0.05)
    dt_record = 0.01
    n_sub = max(1, int(np.ceil(dt_record / dt)))
    dt_act = dt_record / n_sub

    k = fftfreq(N_x, d=1.0/N_x).astype(complex)
    dealias = np.abs(k) <= N_x // 3
    L_op = -nu * k**2

    u0_hat = np.zeros(N_x, dtype=complex)
    for m in range(1, 6):
        amp = rng.standard_normal() + 1j * rng.standard_normal()
        u0_hat[m] += amp
        u0_hat[N_x - m] += np.conj(amp)
    u0_hat *= dealias

    def rhs_hat(u_hat):
        u_phys = np.real(ifft(u_hat))
        nonlin = fft(0.5 * u_phys**2) * dealias
        return L_op * u_hat - 1j * k * nonlin

    U = np.zeros((T, N_x), dtype=np.float32)
    u_hat = u0_hat.copy()
    for t in range(T):
        U[t] = np.real(ifft(u_hat)).astype(np.float32)
        for _ in range(n_sub):
            k1 = rhs_hat(u_hat)
            k2 = rhs_hat(u_hat + 0.5*dt_act*k1)
            k3 = rhs_hat(u_hat + 0.5*dt_act*k2)
            k4 = rhs_hat(u_hat + dt_act*k3)
            u_hat = u_hat + (dt_act/6.0)*(k1+2*k2+2*k3+k4)
            u_hat *= dealias
            if not np.isfinite(u_hat).all():
                print(f'    Diverged at t={t}')
                return U[:t]
    return U

def pca_reduction(U, n_components):
    U_c = U - U.mean(axis=0, keepdims=True)
    n_c = min(n_components, min(U_c.shape)-1)
    _, _, Vt = svd(U_c, full_matrices=False)
    return (U_c @ Vt[:n_c].T).astype(np.float32)

def load_burgers(nu):
    U = simulate_burgers_stable(T=1000, N_x=128, nu=nu, seed=SEED)
    if len(U) < CONTEXT_LEN + 10:
        raise ValueError(f"Burgers nu={nu}: solver too short ({len(U)} steps) for CONTEXT_LEN={CONTEXT_LEN}")
    pca_series = pca_reduction(U, 16)
    return pca_series.T  # (16, T)

# --- Weather (verbatim load_ts from fixed_experiments.ipynb) ---
def load_ts(path):
    df = pd.read_csv(path)
    df = df.select_dtypes(include=[np.number])
    return df.values.astype(np.float32).T  # (C, T)

def load_weather():
    return load_ts(f'{DATA_DIR}/weather.csv')

# --- Assemble eval_context_windows for every system_key in FROZEN_SYSTEMS ---
BURGERS_NU_MAP = {
    "burgers_nu1p0": 1.0, "burgers_nu0p5": 0.5, "burgers_nu0p05": 0.05,
    "burgers_nu0p1": 0.1, "burgers_nu0p005": 0.005, "burgers_nu0p02": 0.02,
    "burgers_nu0p01": 0.01, "burgers_nu2p0": 2.0,
}

eval_context_windows = {}
eval_context_windows["lorenz"] = extract_context_windows(load_lorenz(), n_windows=N_WINDOWS_PER_SYSTEM)
eval_context_windows["rossler"] = extract_context_windows(load_rossler(), n_windows=N_WINDOWS_PER_SYSTEM)
eval_context_windows["harmonic"] = extract_context_windows(load_harmonic(), n_windows=N_WINDOWS_PER_SYSTEM)
eval_context_windows["van_der_pol"] = extract_context_windows(load_vanderpol(), n_windows=N_WINDOWS_PER_SYSTEM)
eval_context_windows["duffing"] = extract_context_windows(load_duffing(), n_windows=N_WINDOWS_PER_SYSTEM)
for key, nu in BURGERS_NU_MAP.items():
    eval_context_windows[key] = extract_context_windows(load_burgers(nu), n_windows=N_WINDOWS_PER_SYSTEM)
weather_data = load_weather()
for h_key in ["weather_h96", "weather_h192", "weather_h336"]:
    # Context window does not depend on horizon (always CONTEXT_LEN=512); all three
    # horizon rows share the same context extraction.
    eval_context_windows[h_key] = extract_context_windows(weather_data, n_windows=N_WINDOWS_PER_SYSTEM)

missing = set(FROZEN_SYSTEMS["system_key"]) - set(eval_context_windows.keys())
assert not missing, f"Missing loaders for: {missing}"
print(f"Context windows assembled for all {len(eval_context_windows)} frozen systems.")


Context windows assembled for all 16 frozen systems.


In [8]:
# ============================================================
# CELL 10 — ARM 1: REPRESENTATION-SPACE DISTANCE
# (fully written; runs once Cells 5, 7, 9 are confirmed working)
# Aggregation matches this project's own evaluate()/single_condition_mae() convention:
# one distance PER WINDOW, then median + IQR across windows -- not a single distance
# computed from mean-pooled features. This mirrors how rel_skill's own MAE numbers
# were aggregated, so the two are computed the same way rather than by two different
# conventions that happen to get compared.
# ============================================================
arm1_median, arm1_iqr = {}, {}
for system_key, windows in eval_context_windows.items():
    per_window_d = []
    for w in windows:
        feat = extract_features(w)
        d = mahalanobis(feat, skew40_centroid, skew40_cov_inv)
        per_window_d.append(d)
    per_window_d = np.array(per_window_d)
    arm1_median[system_key] = float(np.median(per_window_d))
    arm1_iqr[system_key] = float(np.percentile(per_window_d, 75) - np.percentile(per_window_d, 25))

FROZEN_SYSTEMS["arm1_distance"] = FROZEN_SYSTEMS["system_key"].map(arm1_median)
FROZEN_SYSTEMS["arm1_distance_iqr"] = FROZEN_SYSTEMS["system_key"].map(arm1_iqr)
print(FROZEN_SYSTEMS[["system_key", "rel_skill", "arm1_distance", "arm1_distance_iqr"]].to_string(index=False))


     system_key  rel_skill  arm1_distance  arm1_distance_iqr
         lorenz      9.600      14.384669           2.093007
       harmonic      6.700      12.577226           0.123892
        rossler      4.700      28.137375          10.907223
  burgers_nu1p0      3.060      21.260960           2.695784
  burgers_nu0p5      2.540      19.199547           1.440931
 burgers_nu0p05      2.090      18.987317           0.247469
  burgers_nu0p1      1.990      18.601637           0.331100
burgers_nu0p005      1.810      19.511643           0.214387
 burgers_nu0p02      1.860      19.613899           0.245637
 burgers_nu0p01      1.690      19.702017           0.308237
    weather_h96      1.272      35.094789          38.018181
   weather_h192      1.326      35.094789          38.018181
   weather_h336      1.279      35.094789          38.018181
        duffing      1.370      20.903639           3.088962
    van_der_pol      1.330      16.938260           1.009425
  burgers_nu2p0      1.2

In [9]:
# ============================================================
# CELL 11 — ARM 2: MODEL-AGNOSTIC DISTANCE
# (fully written; depends on eval_context_windows and skew40_trajectories from
#  Cells 6/9 having shape (channels, time))
# ============================================================
PATCH_SIZE = 16  # TODO: confirm matches Panda's actual patch size (log states 16;
                  # also printed by A3-style config inspection if you want to double check:
                  # getattr(model.config, 'patch_length', 16))

def dominant_timescale_samples(x_1d):
    x = x_1d - x_1d.mean()
    fft_x = np.fft.rfft(x)
    freqs = np.fft.rfftfreq(len(x))
    power = np.abs(fft_x) ** 2
    power[0] = 0
    peak_idx = np.argmax(power)
    if freqs[peak_idx] == 0:
        return len(x)
    return 1.0 / freqs[peak_idx]

def patch_amplitude_variability(window, patch_size=PATCH_SIZE):
    n_channels, T = window.shape
    n_patches = T // patch_size
    vals = []
    for c in range(n_channels):
        patches = window[c, :n_patches*patch_size].reshape(n_patches, patch_size)
        norms = np.linalg.norm(patches, axis=1)
        vals.append(np.std(np.diff(norms)))
    return float(np.mean(vals))

def arm2_feature_vector(window):
    n_channels = window.shape[0]
    timescales = [dominant_timescale_samples(window[c]) for c in range(window.shape[0])]
    return np.array([
        n_channels,
        np.mean(timescales),
        patch_amplitude_variability(window),
    ])

skew40_arm2 = np.array([arm2_feature_vector(t) for t in skew40_trajectories])
skew40_arm2_mean = skew40_arm2.mean(axis=0)
skew40_arm2_std = skew40_arm2.std(axis=0) + 1e-8

# Same per-window-then-median/IQR convention as Arm 1 (Cell 10), for the same reason:
# match how rel_skill's own MAE numbers were aggregated.
arm2_median, arm2_iqr = {}, {}
for system_key, windows in eval_context_windows.items():
    per_window_d = []
    for w in windows:
        feat = arm2_feature_vector(w)
        z = (feat - skew40_arm2_mean) / skew40_arm2_std
        per_window_d.append(float(np.linalg.norm(z)))
    per_window_d = np.array(per_window_d)
    arm2_median[system_key] = float(np.median(per_window_d))
    arm2_iqr[system_key] = float(np.percentile(per_window_d, 75) - np.percentile(per_window_d, 25))

FROZEN_SYSTEMS["arm2_distance"] = FROZEN_SYSTEMS["system_key"].map(arm2_median)
FROZEN_SYSTEMS["arm2_distance_iqr"] = FROZEN_SYSTEMS["system_key"].map(arm2_iqr)
print(FROZEN_SYSTEMS[["system_key", "rel_skill", "arm1_distance", "arm2_distance", "arm2_distance_iqr"]].to_string(index=False))


     system_key  rel_skill  arm1_distance  arm2_distance  arm2_distance_iqr
         lorenz      9.600      14.384669       0.803531           0.480960
       harmonic      6.700      12.577226       5.470795           0.013133
        rossler      4.700      28.137375       0.437888           0.061652
  burgers_nu1p0      3.060      21.260960      33.182537           0.016067
  burgers_nu0p5      2.540      19.199547      33.174274           0.026015
 burgers_nu0p05      2.090      18.987317      33.138863           0.020257
  burgers_nu0p1      1.990      18.601637      33.151845           0.023408
burgers_nu0p005      1.810      19.511643      33.138863           0.005899
 burgers_nu0p02      1.860      19.613899      33.145157           0.013374
 burgers_nu0p01      1.690      19.702017      33.138863           0.025963
    weather_h96      1.272      35.094789      45.972476           0.015795
   weather_h192      1.326      35.094789      45.972476           0.015795
   weather_h

In [10]:
# ============================================================
# CELL 12 — CORRELATION AND PRE-REGISTERED DECISION RULE (fully written)
# ============================================================
rho1, p1 = spearmanr(FROZEN_SYSTEMS["arm1_distance"], FROZEN_SYSTEMS["rel_skill"])
rho2, p2 = spearmanr(FROZEN_SYSTEMS["arm2_distance"], FROZEN_SYSTEMS["rel_skill"])

print(f"Arm 1 (representation-space): rho = {rho1:.3f}, p = {p1:.4f}")
print(f"Arm 2 (model-agnostic):       rho = {rho2:.3f}, p = {p2:.4f}")
print()

def arm_verdict(rho, label):
    if rho <= -0.5:
        print(f"{label}: SUPPORTS H-dist (rho <= -0.5, correct direction)")
        return "support"
    else:
        print(f"{label}: does not meet H-dist threshold")
        return "no_support"

v1 = arm_verdict(rho1, "Arm 1")
v2 = arm_verdict(rho2, "Arm 2")

print()
if v1 == "support" and v2 == "support":
    print("OVERALL: STRONG support for H-dist (both arms agree).")
elif v1 == "support" or v2 == "support":
    print("OVERALL: WEAK support only — one arm meets threshold, the other does not.")
    if v1 == "support" and v2 == "no_support":
        print("  -> Arm 1-only support is the pattern most consistent with H-confound-circularity.")
        print("     Treat with the same caution as Section 8's Burgers eDMD anomaly: flag, don't conclude.")
else:
    print("OVERALL: NO support for H-dist in either arm.")
    print("  This would be a fifth consecutive null on a candidate mechanism.")


Arm 1 (representation-space): rho = -0.602, p = 0.0137
Arm 2 (model-agnostic):       rho = -0.501, p = 0.0478

Arm 1: SUPPORTS H-dist (rho <= -0.5, correct direction)
Arm 2: SUPPORTS H-dist (rho <= -0.5, correct direction)

OVERALL: STRONG support for H-dist (both arms agree).


## Caveats to carry into interpretation, whatever the result

- **n≈15, several cells at n_windows=8.** A positive finding here motivates
  confirmatory reruns of those specific cells (the continuum systems, individual
  Burgers ν values), it does not substitute for them — same logic as the
  heterogeneity collapse.
- **ETTh1/ETTh2 excluded from the primary correlation** since Experiment 8 found
  no consistent-direction significant result on either across horizons; forcing
  a single rel_skill number onto them would misrepresent a null result as a data
  point.
- **Arm 1 / Arm 2 disagreement is not a coin flip to resolve by picking the arm
  you like** — per the decision rule, disagreement is read as evidence against
  H-dist (or against Arm 1's independence from circularity), not as 1-of-2 support.
- **This experiment does not distinguish "close to skew40" from "close to
  whichever subset of skew40 systems resemble each eval system,"** e.g. Lorenz-
  family vs. other chaotic systems, the same sub-cluster issue flagged in
  Experiment 4. A follow-up nearest-neighbour breakdown (not built here) would be
  needed to unpack a positive result further.
- **The skew40 extraction in Cell 6 is unverified against a live schema.** Confirm
  the column names and axis order before trusting anything downstream of it.
- **Single trajectory per eval system (n_windows=1 in Cell 9).** This mirrors the
  single-trajectory convention used throughout the log's OOD tables, but it means
  Arm 1/Arm 2 distances are each a point estimate, not a distribution. If a result
  here looks borderline, increasing n_windows per system (multiple context windows
  per trajectory, or multiple seeded trajectories per system) is the first thing
  to try before trusting a borderline correlation.


In [11]:
# ============================================================
# CELL 13 — DIAGNOSTIC A: PERIOD-COUNT MISMATCH CHECK
# skew40 trajectories are each cut to ~40 characteristic periods (per-system
# timescale). This notebook's synthetic-ODE eval windows were NOT built to match
# that convention. Checking how many periods each 512-step eval window actually
# spans, using the same dominant_timescale_samples() already defined in Cell 11.
# ============================================================
SYNTHETIC_ODE_SYSTEMS = ["lorenz", "rossler", "harmonic", "duffing", "van_der_pol"]

print(f'{"system":15s} {"mean period (samples)":>22s} {"periods in 512-step window":>28s}')
for key in SYNTHETIC_ODE_SYSTEMS:
    window = eval_context_windows[key][0]  # first window, single trajectory
    per_channel_periods = [dominant_timescale_samples(window[c]) for c in range(window.shape[0])]
    mean_period = float(np.mean(per_channel_periods))
    n_periods = CONTEXT_LEN / mean_period
    flag = "  <-- far from 40" if not (20 <= n_periods <= 80) else ""
    print(f'{key:15s} {mean_period:22.2f} {n_periods:28.2f}{flag}')

print()
print("If any system's period count is wildly different from ~40 (skew40's own "
      "convention), Arm 1's distance for that system is at least partly measuring "
      "sampling-density mismatch, not distributional similarity -- the same failure "
      "mode Experiment 31 found for the structure statistic under downsampling.")

system           mean period (samples)   periods in 512-step window
lorenz                           64.00                         8.00  <-- far from 40
rossler                         142.22                         3.60  <-- far from 40
harmonic                        128.00                         4.00  <-- far from 40
duffing                         128.00                         4.00  <-- far from 40
van_der_pol                     170.67                         3.00  <-- far from 40

If any system's period count is wildly different from ~40 (skew40's own convention), Arm 1's distance for that system is at least partly measuring sampling-density mismatch, not distributional similarity -- the same failure mode Experiment 31 found for the structure statistic under downsampling.


In [12]:
# ============================================================
# CELL 14 — DIAGNOSTIC B: FAMILY-COLLAPSED ROBUSTNESS CHECK
# Collapses the pseudo-replicated rows (8 Burgers-nu rows -> 1, 3 Weather-horizon
# rows -> 1, via median) to get a fairer effective-n test. This is NOT a
# replacement for the pre-registered n=16 result -- it's the check that determines
# whether that result is being driven by genuine cross-system signal or mostly by
# within-family duplication.
# ============================================================
FAMILY_MAP = {
    "lorenz": "lorenz", "harmonic": "harmonic", "rossler": "rossler",
    "duffing": "duffing", "van_der_pol": "van_der_pol",
    **{k: "burgers_family" for k in BURGERS_NU_MAP.keys()},
    "weather_h96": "weather_family", "weather_h192": "weather_family", "weather_h336": "weather_family",
}

collapsed = FROZEN_SYSTEMS.copy()
collapsed["family"] = collapsed["system_key"].map(FAMILY_MAP)
collapsed_agg = collapsed.groupby("family").agg(
    rel_skill=("rel_skill", "median"),
    arm1_distance=("arm1_distance", "median"),
    arm2_distance=("arm2_distance", "median"),
    n_rows_collapsed=("system_key", "count"),
).reset_index()

print(collapsed_agg.to_string(index=False))
print(f"\nEffective n after family collapse: {len(collapsed_agg)} (vs. {len(FROZEN_SYSTEMS)} nominal rows)")

rho1_c, p1_c = spearmanr(collapsed_agg["arm1_distance"], collapsed_agg["rel_skill"])
rho2_c, p2_c = spearmanr(collapsed_agg["arm2_distance"], collapsed_agg["rel_skill"])
print(f"\nFamily-collapsed Arm 1: rho = {rho1_c:.3f}, p = {p1_c:.4f}")
print(f"Family-collapsed Arm 2: rho = {rho2_c:.3f}, p = {p2_c:.4f}")
print()
print("Compare against the nominal n=16 result (Cell 12). If these collapse toward")
print("zero or lose significance, the n=16 result was substantially a pseudo-")
print("replication artifact, not independent cross-family evidence.")

        family  rel_skill  arm1_distance  arm2_distance  n_rows_collapsed
burgers_family      1.925      19.562771      33.145725                 8
       duffing      1.370      20.903639       5.495328                 1
      harmonic      6.700      12.577226       5.470795                 1
        lorenz      9.600      14.384669       0.803531                 1
       rossler      4.700      28.137375       0.437888                 1
   van_der_pol      1.330      16.938260       5.462141                 1
weather_family      1.279      35.094789      45.972476                 3

Effective n after family collapse: 7 (vs. 16 nominal rows)

Family-collapsed Arm 1: rho = -0.643, p = 0.1194
Family-collapsed Arm 2: rho = -0.607, p = 0.1482

Compare against the nominal n=16 result (Cell 12). If these collapse toward
zero or lose significance, the n=16 result was substantially a pseudo-
replication artifact, not independent cross-family evidence.
